In [ ]:
from time import sleep
from typing import TypedDict, Literal

from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_deepseek import ChatDeepSeek
from rich import print

from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を宣言
class OverAllState(TypedDict):
    topic: str
    poem: str
    is_approved: bool


#2. ノードを宣言
def approve_node(state: OverAllState) -> Command[Literal["llm_node", "default_node"]]:
    is_approved = interrupt("モデルの呼び出しに同意しますか？")
    goto = "llm_node" if is_approved else "default_node"
    return Command(
        goto=goto,
        update={"is_approved": is_approved}
    )


def llm_node(state: OverAllState) -> OverAllState:
    topic = state["topic"]
    res = model.invoke([HumanMessage(content=f"{topic}をテーマにした俳句を書いてください。俳句本文だけでよく、鑑賞は不要です")]).content

    return {
        "poem": res
    }


def default_node(state: OverAllState) -> OverAllState:
    return {
        "poem": "リクエストが拒否されました"
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("approve_node", approve_node)
builder.add_node("llm_node", llm_node)
builder.add_node("default_node", default_node)

builder.add_edge(START, "approve_node")
builder.add_edge("llm_node", END)
builder.add_edge("default_node", END)

#4. チェックポインターバックエンドを追加
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import display

display(graph)

#5. 初回実行 -> 中断をトリガー
config = {"configurable": {"thread_id": "234"}}
interrupt_res = graph.invoke({"topic": "菊"}, config=config)
print(interrupt_res)



In [ ]:
#6. 人による承認
user_approved = input("モデルの呼び出しに同意しますか？(y/n)").strip().lower() == 'y'
# グラフを再度呼び出す
approved_res = graph.invoke(Command(resume=user_approved), config=config)
print(approved_res)


In [ ]:
#7. 承認されなかった場合
config1 = {"configurable": {"thread_id": "456"}}
interrupt_res = graph.invoke({"topic": "牡丹"}, config=config1)
print(interrupt_res)

user_approved = input("モデルの呼び出しに同意しますか？(y/n)").strip().lower() == 'y'

# グラフを再度呼び出す
approved_res = graph.invoke(Command(resume=user_approved), config=config1)
print(approved_res)